My Dad got me a fancy MacBook for my birthday so I figure I'd give MLX a try.

In [1]:
%pip install mlx-lm

Note: you may need to restart the kernel to use updated packages.


In [2]:
from mlx_lm import load, generate

model, tokenizer = load("mlx-community/gpt-oss-20b-MXFP4-Q8")

messages = [{"role": "user", "content": "Write a story about Einstein."}]
prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
)
response = generate(
    model,
    tokenizer,
    prompt=prompt,
    max_tokens=256,
)
response

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

'<|channel|>analysis<|message|>We need to write a story about Einstein. The user didn\'t specify length or style. We can write a creative narrative, perhaps a fictionalized account of Einstein\'s life, or a story that includes Einstein as a character. Could be a short story, maybe a whimsical or imaginative take. The user just says "Write a story about Einstein." So we can choose a creative angle. Maybe a story where Einstein is a character in a modern setting, or a story about his childhood, or a story about his relativity concept. We can incorporate some of his famous quotes or ideas. We can also make it a story about a young student who meets Einstein. Or a story about Einstein\'s time in exile. Or a story about a child who discovers relativity through a conversation with Einstein. Or a story about Einstein\'s love for music. Or a story about Einstein\'s time in America. Or a story about Einstein\'s relationship with his wife. Or a story about Einstein\'s involvement in the Manhatta

In [3]:
%pip install datasets

Note: you may need to restart the kernel to use updated packages.


`GAIR/LIMO-v2` has nice, clean questions and answers. It's only math but that's okay for now.

In [4]:
from datasets import load_dataset

ds = load_dataset("GAIR/LIMO-v2", split="train")
ds["question"][:5]

['Given \\( m = n^{4} + x \\), where \\( n \\) is a natural number and \\( x \\) is a two-digit positive integer, what value of \\( x \\) will make \\( m \\) a composite number?',
 'In triangle \\( ABC \\), side \\( AC \\) is the largest. Points \\( M \\) and \\( N \\) on side \\( AC \\) are such that \\( AM = AB \\) and \\( CN = CB \\). It is known that angle \\( \\angle NBM \\) is three times smaller than angle \\( \\angle ABC \\). Find \\( \\angle ABC \\).',
 'Let $a,$ $b,$ $c,$ $d$ be real numbers, none of which are equal to $-1,$ and let $\\omega$ be a complex number such that $\\omega^3 = 1$ and $\\omega \\neq 1.$  If\n\\[\\frac{1}{a + \\omega} + \\frac{1}{b + \\omega} + \\frac{1}{c + \\omega} + \\frac{1}{d + \\omega} = \\frac{2}{\\omega},\\]then find\n\\[\\frac{1}{a + 1} + \\frac{1}{b + 1} + \\frac{1}{c +1} + \\frac{1}{d + 1}.\\]',
 'Sasha wrote the numbers $7, 8, 9, \\ldots, 17$ on the board and then erased one or more of them. It turned out that the remaining numbers on the bo

I want to do a "tree of generations" method so we can get advantages for prefixes of generations, not just entire generations. I think this is important because I believe generations that follow paths that are too different make it hard to contrast what exactly went right or wrong between generations.

In [18]:
from mlx_lm import stream_generate
from mlx_lm.sample_utils import make_sampler
from mlx_lm.models.cache import make_prompt_cache
import copy

FINAL_HEADER = "<|channel|>final<|message|>"
ANALYSIS_HEADER = "<|channel|>analysis<|message|>"
STEP_DELIM = "\n\n"

def generate_tree(
    prompt,
    prompt_cache=None,
    branching_factor=2,
    max_depth=8,
    *args,
    **kwargs,
):
    if max_depth < 1:
        raise ValueError("max_depth must be at least 1")

    if max_depth == 1:
        return {
            "step": generate(
                model,
                tokenizer,
                prompt=prompt,
                prompt_cache=prompt_cache,
                *args,
                **kwargs,
            ),
            "branches": None,
        }

    branches = []
    for _ in range(branching_factor):
        stream = stream_generate(
            model,
            tokenizer,
            prompt=prompt,
            prompt_cache=copy.deepcopy(prompt_cache),
            *args,
            **kwargs,
        )

        step = ""
        branch = False
        for chunk in stream:
            step += chunk.text

            if FINAL_HEADER in prompt + step:
                pass

            elif ANALYSIS_HEADER in prompt + step:
                if STEP_DELIM in step:
                    step = step[:step.index(STEP_DELIM) + len(STEP_DELIM)]
                    branch = True
                    break
        
        branches.append({
            "step": step,
            "subtree": generate_tree(
                prompt + step,
                branching_factor=branching_factor,
                max_depth=max_depth - 1,
                *args,
                **kwargs,
            ) if branch else None,
        })
    return branches

messages = [{"role": "user", "content": ds["question"][0]}]
prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
)
prompt_cache = make_prompt_cache(model)
sampler = make_sampler(temp=1.0, top_p=0.95)
tree = generate_tree(
    prompt,
    prompt_cache=prompt_cache,
    max_depth=6,
    sampler=sampler,
    max_tokens=256,
)
tree

[{'step': '<|channel|>analysis<|message|>We need to find x such that m is composite for all n? Or for some n? The phrasing: Given m = n^4 + x, where n is natural number and x is a two-digit positive integer, what value of x will make m a composite number? It might ask find all two-digit x such that n^4 + x is composite for all natural n? Or maybe find x such that m is composite for some n? Problem ambiguous.\n\n',
  'subtree': [{'step': 'Could be like find x such that n^4 + x always composite. For some n? Let’s analyze: n^4 is always a fourth power; mod small primes maybe. We look for x that is a multiple of something that yields a composite n^4 + x divisible by some fixed factor.\n\n',
    'subtree': [{'step': "We know n^4 mod 5? Since n^4 mod 5 is 0 or 1? Actually by Fermat's little theorem, n^4 mod 5 is 0 if 5 divides n, else 1 (since n^4 ≡ 1 mod 5 for n coprime to 5). So n^4 mod 5 ∈ {0,1}. For x = 4? Then n^4 + 4 ≡ 4 mod 5 if n^4 mod5=0, else 0 mod5. So sometimes divisible by 5. No